# 1  定义工具

In [14]:
import os

from langchain_chroma import Chroma
from langchain_community.tools.tavily_search import TavilySearchResults

# 定义 AVILY_KEY 密钥
os.environ["TAVILY_API_KEY"] = "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"
# 查询 Tavily 搜索 API
search = TavilySearchResults(max_results=1)
# 执行查询
res = search.invoke("2月19日上海天气怎么样")
print(res)

[{'title': '上海2月天气怎么样？气温、穿搭建议与春节旅行指南 - Trip.com', 'url': 'https://my.trip.com/guide/info/%E4%B8%8A%E6%B5%B72%E6%9C%88%E5%A4%A9%E6%B0%94.html', 'content': 'December and February, temperatures will still be cold, but won\'t go below freezing. Highs average at 10°C (approx. 50°C), and lows average at 3°C (37°F). [...] Lihat lagi\n\n## 上海的四季\n\n春季（3月–5月）\n\n气温逐渐回升，但天气多变，时常阴雨连绵。3月仍偏凉，4–5月体感逐渐舒适。\n\n夏季（6月–9月）\n\n炎热、闷湿，是一年中最难熬的季节。高温常伴随雷阵雨和台风，体感温度较高。\n\n秋季（10月–11月）\n\n被认为是上海最适合旅游的时间。天气晴朗、湿度下降、气温舒适，非常适合城市漫步和拍照。\n\n冬季（12月–2月）\n\n上海的冬季偏冷且湿润，1月通常是全年最冷的月份。虽然气温不算极低，但湿冷体感明显，需要充分保暖。\n\n🎁 搜索飞往上海的最便宜航班，预订最棒的上海酒店！使用中国 eSIM快速连接。\n\n## 上海2月天气：冬季尾声，寒意仍在\n\n2月的上海正处在冬季向春季过渡的阶段。相比1月，气温略有回升，但“湿冷”体感依旧明显。阴天和小雨仍较常见，早晚温差相对明显，尤其在下雨或刮风时，会感觉比实际气温更冷。\n\n2月也是上海一年中节日氛围最浓厚的月份之一。多数年份的春节都会落在2月，城市装饰、商场布置和年味十足的街景，为冬季的上海增添了不少活力。对游客来说，这是一个既能体验城市文化、又能避开旺季人潮的好时机。\n\n无论是在外滩散步、走进博物馆，还是在商场和咖啡馆中“躲冷”，2月的上海都适合慢节奏探索。\n\n### 2月天气速览\n\n| 类别 | 详情 |\n --- |\n| 2月季节 | 冬季（向春季过渡） |\n| 温度范围 | 3°C–11°C / 37°F–52°F |\n| 雨量 | 中等 – 月平均降雨量约 50–70 毫米

# 2 定义Retriever

In [15]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import dotenv
dotenv.load_dotenv()
from langchain_chroma import Chroma

# 1. 提供一个大模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

embedding_model = OpenAIEmbeddings()

# 2.加载HTML内容为一个文档对象
loader = WebBaseLoader("https://zh.wikipedia.org/wiki/%E7%8C%AB")
docs = loader.load()
#print(docs)

# 3.分割文档
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = splitter.split_documents(docs)

db_path = './asset/chroma-5'

# 4.向量化 得到向量数据库对象
vector = Chroma.from_documents(documents, embedding_model,persist_directory=db_path)

# 5.创建检索器
retriever = vector.as_retriever()

# 测试检索结果
# print(retriever.invoke("猫的特征")[0])

# 3 创建工具、工具集

注意，这里是另外一种思路不同，在7.5的最后一个例子中，retriever是将向量数据库中检索到的最匹配的向量变回文本之后，再输入给提示词模版中；而这里是将retriever封装成一个工具放入到工具集中。

In [16]:
from langchain.tools.retriever import create_retriever_tool

# 创建一个工具来检索文档
retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="wiki_search",
    description="搜索维基百科",
)

# 构建工具集
tools = [search, retriever_tool]

# 4 语言模型调用工具

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 获取大模型
model = ChatOpenAI(model="gpt-4o-mini")

# 模型绑定工具
model_with_tools = model.bind_tools(tools)

# 根据输入自动调用工具
messages = [HumanMessage(content="今天上海天气怎么样")]
response = model_with_tools.invoke(messages)
print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")


ContentString: 
ToolCalls: [{'name': 'tavily_search_results_json', 'args': {'query': '上海天气'}, 'id': 'call_FdpbPksgfdFmDloqxd2cQluF', 'type': 'tool_call'}]


# 5 创建Agent程序(使用通用方式)

传统方式有定义好的提示词模版，而通用方式则是需要自定义提示词模版，这里使用“hwchase17/openai-functions-agent”现成的，这个6.2有讲过。

In [6]:
from langchain import hub
prompt = hub.pull("hwchase17/openai-functions-agent")

print(prompt.messages)

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}), MessagesPlaceholder(variable_name='chat_history', optional=True), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), MessagesPlaceholder(variable_name='agent_scratchpad')]


In [7]:
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

# 创建Agent对象
agent = create_tool_calling_agent(model, tools, prompt)

# 创建AgentExecutor对象
agent_executor = AgentExecutor(agent=agent, tools=tools,verbose=True)

# 6 运行Agent

In [18]:
print(agent_executor.invoke({"input": "猫的特征"}))

猫的特征包括以下几个方面：

1. **感官敏锐**：猫的感官适合狩猎，尤其是在听觉、视觉、嗅觉、味觉和触觉方面非常敏锐。

2. **视觉特性**：
   - 猫在夜间的视觉能力是人类的六倍，能够在微弱的光线下看见物体，但在白天的视觉不如人类。
   - 猫的眼睛能够放大光线，使其在黑暗中有强大的导航能力。即使在光线昏暗的环境中，猫也能自如活动。
   - 它们的眼睛包含一种特殊的光反射层（Tapetum Lucidum），能提高在黑暗中的视力。
   - 猫的视野范围较广，双眼视野可达285度，但对三原色的辨识能力较差。

3. **嗅觉和味觉**：
   - 猫的嗅觉非常灵敏，对气味的敏感程度比人类更高。
   - 需要注意的是，猫对甜味不敏感，因为它们缺乏一种甜味受体基因。

4. **触觉**：
   - 猫的须毛（触须）分布在脸部和身体的关键位置，帮助它们感知周围环境和判断空间。

5. **其他特征**：
   - 猫有第三眼睑，用于保护和湿润眼睛。
   - 猫的行为和情绪会影响它们的瞳孔大小，紧张时瞳孔放大，放松时瞳孔缩小。

这些特征使猫在狩猎和自我保护方面表现出色，也使它们成为了非常适合夜间活动的捕食者。

> Finished chain.
{'input': '猫的特征', 'output': '猫的特征包括以下几个方面：\n\n1. **感官敏锐**：猫的感官适合狩猎，尤其是在听觉、视觉、嗅觉、味觉和触觉方面非常敏锐。\n\n2. **视觉特性**：\n   - 猫在夜间的视觉能力是人类的六倍，能够在微弱的光线下看见物体，但在白天的视觉不如人类。\n   - 猫的眼睛能够放大光线，使其在黑暗中有强大的导航能力。即使在光线昏暗的环境中，猫也能自如活动。\n   - 它们的眼睛包含一种特殊的光反射层（Tapetum Lucidum），能提高在黑暗中的视力。\n   - 猫的视野范围较广，双眼视野可达285度，但对三原色的辨识能力较差。\n\n3. **嗅觉和味觉**：\n   - 猫的嗅觉非常灵敏，对气味的敏感程度比人类更高。\n   - 需要注意的是，猫对甜味不敏感，因为它们缺乏一种甜味受体基因。\n\n4. **触觉**：\n   - 猫的须毛（触须）分布在脸部和身体的关键位置，帮助它们感知周围环境和判断空间。\n\n5. **其他特征*

In [9]:
print(agent_executor.invoke({"input": "今天上海天气怎么样"}))

今天上海的天气状况如下：

- 当前温度：约 12°C (54°F)，感觉温度也为 12°C。
- 天气情况：多云。
- 风速：东南偏南 4 英里/小时。
- 空气质量：不健康。

你可以查看更详细的天气预报信息 [这里](https://www.accuweather.com/zh/cn/shanghai/106577/weather-forecast/106577)。

> Finished chain.
{'input': '今天上海天气怎么样', 'output': '今天上海的天气状况如下：\n\n- 当前温度：约 12°C (54°F)，感觉温度也为 12°C。\n- 天气情况：多云。\n- 风速：东南偏南 4 英里/小时。\n- 空气质量：不健康。\n\n你可以查看更详细的天气预报信息 [这里](https://www.accuweather.com/zh/cn/shanghai/106577/weather-forecast/106577)。'}


# 7 添加记忆

In [19]:
from langchain_community.chat_message_histories import ChatMessageHistory

from langchain_core.chat_history import BaseChatMessageHistory

from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

# 调取指定session_id对应的memory
def get_session_history(session_id: str) -> BaseChatMessageHistory:

    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    return store[session_id]

agent_with_chat_history = RunnableWithMessageHistory(
    runnable=agent_executor,
    get_session_history=get_session_history,
    input_messages_key="input",
    # history_messages_key="chat_history",
)

response = agent_with_chat_history.invoke(
    {"input": "Hi，我的名字是Cyber"},
    config={"configurable": {"session_id": "123"}},
)

print(response)

你好，Cyber！很高兴见到你！有什么我可以帮助你的吗？

> Finished chain.
{'input': [HumanMessage(content='Hi，我的名字是Cyber', additional_kwargs={}, response_metadata={})], 'output': '你好，Cyber！很高兴见到你！有什么我可以帮助你的吗？'}


In [20]:
response = agent_with_chat_history.invoke(
    {"input": "我叫什么名字?"},
    config={"configurable": {"session_id": "123"}},
)

print(response)

你的名字是Cyber！如果你有其他问题或需要帮助的地方，请告诉我！

> Finished chain.
{'input': [HumanMessage(content='Hi，我的名字是Cyber', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，Cyber！很高兴见到你！有什么我可以帮助你的吗？', additional_kwargs={}, response_metadata={}), HumanMessage(content='我叫什么名字?', additional_kwargs={}, response_metadata={})], 'output': '你的名字是Cyber！如果你有其他问题或需要帮助的地方，请告诉我！'}


此时换了id，所以换了聊天历史。

In [21]:
response = agent_with_chat_history.invoke(
    {"input": "我叫什么名字?"},
    config={"configurable": {"session_id": "4566"}},
)
print(response)

抱歉，我无法知道你叫做什么名字。你可以告诉我你的名字吗？

> Finished chain.
{'input': [HumanMessage(content='我叫什么名字?', additional_kwargs={}, response_metadata={})], 'output': '抱歉，我无法知道你叫做什么名字。你可以告诉我你的名字吗？'}
